In [5]:
import os

os.environ['RUBIN_SIM_DATA_DIR'] = '/pscratch/sd/r/rhlozek/rubin_sim_data'
os.environ['RUBIN_SIM_DATA_DIR'] 
import numpy as np
import matplotlib.pyplot as plt
import healpy as hp
import pandas as pd
import rubin_sim
import rubin_sim.maf as maf
#from rubin_sim.data import get_baseline
print(rubin_sim.__version__)
from os.path import splitext, basename
from rubin_scheduler.scheduler.utils import SkyAreaGenerator

0.1.dev1813+gf6a2d0c


# Mean z metric

In [7]:
from rubin_sim.maf.metrics.cosmology_summary_metrics import MultibandMeanzBiasMetric
from rubin_sim.maf.metrics.uniformity_metrics import MultibandExgalM5
from rubin_sim.maf.metrics.weak_lensing_systematics_metric import RIZDetectionCoaddExposureTime, ExgalM5WithCuts
from rubin_sim.maf.metrics.exgal_m5 import ExgalM5

nside = 32

sim_list = [
    '/pscratch/sd/r/rhlozek/rubin_sim_data/sim_baseline/baseline_v3.3_10yrs.db',
  #     '/pscratch/sd/r/rhlozek/rubin_sim_data/noroll/noroll_v3.3_10yrs.db',
    #   '/pscratch/sd/b/beckermr/v3.4_sims_rubin/roll_uniform_mjdp0_v3.4_10yrs.db',
    #   '/pscratch/sd/b/beckermr/v3.4_sims_rubin/baseline_v3.4_10yrs.db'
           ]
name_list = [splitext(basename(sim))[0] for sim in sim_list]

years = range(1, 11) 
surveyAreas = SkyAreaGenerator(nside=nside)
map_footprints, map_labels = surveyAreas.return_maps()
slicer = maf.HealpixSubsetSlicer(
    nside=nside, 
    hpid=np.where(map_labels == "lowdust")[0],#hpid=np.arange(hp.nside2npix(nside)), 
    use_cache=False)

results_allruns= {}
for opsim_fname, run_name in zip(sim_list, name_list):

    results_allyears = np.zeros((len(years), ))
    # loop over years 
    for iy, year in enumerate(years):
        print('year', year)

        days = year*365.25
        constraint_str = 'note not like "DD%" and night <= XX and note not like "twilight_near_sun" '
        constraint_str = constraint_str.replace('XX','%d'%days)

        
        metric = MultibandExgalM5() 
        summary_metrics = [MultibandMeanzBiasMetric(year=year, 
                                                    metric_name='MultibandMeanzBias')]
        depth_map_bundles = [maf.MetricBundle(
            metric=metric,
            slicer=slicer,
            constraint=constraint_str,
            run_name=run_name,
            summary_metrics=summary_metrics
        )]
        bd = maf.metricBundles.make_bundles_dict_from_list(depth_map_bundles)
        bgroup = maf.MetricBundleGroup(bd, opsim_fname)
        bgroup.run_all()
        results_allyears[iy] = bd[list(bd.keys())[0]].summary_values['MultibandMeanzBias']
        
    results_allruns[run_name] = results_allyears

    

Healpix slicer using NSIDE=32, approximate resolution 109.935565 arcminutes
year 1


/pscratch/sd/r/rhlozek/rubin_sim/rubin_sim/maf/maps/dust_map.py:46: UserWarning: Slicer value of nside 32 different from map value 128, using slicer value
  warnings.warn(


Failed at slice_point {'idxs': array([112769, 135813, 123278, 193575,  16111, 165261, 131556,  16098,
       110441, 130904, 131506, 197852, 144187, 164558, 129896,  16124,
       117751, 193554, 144237, 177314, 171685, 177264, 167127,  16095,
        16085, 110416,  16082, 197802, 165211, 148805, 120156, 188587,
       196862, 129096, 129918, 196912, 164508, 130954, 120106, 171707,
       167149, 172644, 129362, 188531, 172694, 129340, 153125, 153175,
       120108, 120158, 197801, 197851, 147536, 134167, 111953, 156302,
       156101, 136772, 200448, 147486, 134117, 161986, 136822, 133255,
       161936, 200496, 133305, 156252, 145820, 203026, 194116, 194066,
       136731, 149438, 149388, 133256, 133306, 136781]), 'slice_point': {'sid': 4580, 'nside': 32, 'ra': 1.7916895602504286, 'dec': 0.2526802551420786, 'gall': 3.4889534288849355, 'galb': 0.11116825198239613, 'ebv': 0.18343958258628845}}, sid 4580


NotImplementedError: Please implement your metric calculation.

NameError: name 'stop' is not defined

# Area at risk metric

In [ ]:
from rubin_sim.maf.metrics.cosmology_summary_metrics import UniformAreaFoMFractionMetric
from rubin_sim.maf.metrics.uniformity_metrics import NestedRIZExptimeExgalM5Metric
from rubin_sim.maf.metrics.weak_lensing_systematics_metric import RIZDetectionCoaddExposureTime, ExgalM5WithCuts
from rubin_sim.maf.metrics.exgal_m5 import ExgalM5

nside = 64

sim_list = [
    '/pscratch/sd/r/rhlozek/rubin_sim_data/sim_baseline/baseline_v3.3_10yrs.db',
       '/pscratch/sd/r/rhlozek/rubin_sim_data/noroll/noroll_v3.3_10yrs.db',
       '/pscratch/sd/b/beckermr/v3.4_sims_rubin/roll_uniform_mjdp0_v3.4_10yrs.db',
    #   '/pscratch/sd/b/beckermr/v3.4_sims_rubin/baseline_v3.4_10yrs.db'
           ]
name_list = [splitext(basename(sim))[0] for sim in sim_list]

years = range(1, 11) 
surveyAreas = SkyAreaGenerator(nside=nside)
map_footprints, map_labels = surveyAreas.return_maps()
slicer = maf.HealpixSubsetSlicer(
    nside=nside, 
    hpid=np.where(map_labels == "lowdust")[0],#hpid=np.arange(hp.nside2npix(nside)), 
    use_cache=False)

results_allruns= {}
for opsim_fname, run_name in zip(sim_list, name_list):

    results_allyears = np.zeros((len(years), ))
    # loop over years 
    for iy, year in enumerate(years):
        print('year', year)

        days = year*365.25
        constraint_str = 'note not like "DD%" and night <= XX and note not like "twilight_near_sun" '
        constraint_str = constraint_str.replace('XX','%d'%days)

        
        metric = NestedRIZExptimeExgalM5Metric(
            depth_cut=25.0 # what depth cuts to apply year after year?
        ) 
        summary_metrics = [UniformAreaFoMFractionMetric(
            nside=nside,
            verbose=True, 
            metric_name='FoMRatio'
        )]
        depth_map_bundles = [maf.MetricBundle(
            metric=metric,
            slicer=slicer,
            constraint=constraint_str,
            run_name=run_name,
            summary_metrics=summary_metrics
        )]
        bd = maf.metricBundles.make_bundles_dict_from_list(depth_map_bundles)
        bgroup = maf.MetricBundleGroup(bd, opsim_fname)
        bgroup.run_all()
        results_allyears[iy] = bd[list(bd.keys())[0]].summary_values['FoMRatio']
        
    results_allruns[run_name] = results_allyears

    

In [ ]:

fig, ax = plt.subplots(1, 1, figsize=(7, 7), sharex=True)

colors = ['orange', 'blue', 'black', 'red']
for i, run_name in enumerate(results_allruns.keys()):
    ax.plot(years, results_allruns[run_name], label=run_name, marker='o', color=colors[i])
ax.legend()

ax.set_xlabel('Years')
ax.set_ylabel('FOM ratio')

In [ ]:
stop

# Tomographic Clustering Sigma8 bias Metric

In [ ]:
from rubin_sim.maf.metrics.uniformity_metrics import NestedLinearMultibandModelMetric
from rubin_sim.maf.metrics.cosmology_summary_metrics import TomographicClusteringSigma8biasMetric

In [ ]:
from rubin_sim.maf.metrics.tomography_models import DENSITY_TOMOGRAPHY_MODEL
# this contains the current model.
# the first set of keys are the years (year1, ..., year10) since this would change typical depth and galaxy catalog cuts.
# in what follows we have 5 tomographic bins.
# the second nested dictionary has the following:
# sigma8square_model is the fiducial sigma8^2 value used in CCL for the theory predictions
# poly1d_coefs_loglog is a polynomial (5th degree) describing the angular power spectra (in log log space) in the 5 tomographic bins considered, thus has shape (5, 6)
# lmax contains the lmax limits to sum the Cells over when calculating sigma8 for each tomographic bin. thus is it of shape (5, )
# dlogN_dm5 contains the derivatives of logN wrt m5 calculated in Qianjun & Jeff's simulations. It is an array of 5 dictionaries (5 = the tomographic bins)
# each dictionary must have keys that are the lsst bands. If some are missing they are ignored in the linear model.
# they are the ones which will be fed to LinearMultibandModelMetric. Everything else above is going into the modeling 
# The notebook I used to make this dictionary is https://github.com/ixkael/ObsStrat/blob/meanz_uniformity_maf/code/meanz_uniformity/romanrubinmock_for_sigma8tomography.ipynb

In [ ]:
# a simple wrapper around the metrics, to store the results, but not critically needed
def extract_sigma8_tomography_metric(
    opsim_fname, run_fname,
    years, 
    percentage_uncorrected,
    density_tomography_model, 
    lmin = 10,
    mag_range_tolerated=1.0,
    n_filters = 6, 
    extinction_cut = 0.2, # sky cuts
    nside=32,
    convert_to_sigma8=True
):
    surveyAreas = SkyAreaGenerator(nside=nside)
    map_footprints, map_labels = surveyAreas.return_maps()
    slicer = maf.HealpixSubsetSlicer(
        nside=nside, 
        hpid=np.where(map_labels == "lowdust")[0],#hpid=np.arange(hp.nside2npix(nside)), 
        use_cache=False)
    
    # prepare empty arrays to fill in the results
    n_bins = 5 # set to 5
    results_spuriousdensitypower = np.zeros((len(years), n_bins))
    results_sigma8_squared_bias = np.zeros((len(years), ))
    # loop over years
    all_depth_map_bundles = []
    for iy, year in enumerate(years):
        print('year', year)
        
        # constraints
        days = year*365.25
        constraint_str = 'note not like "DD%" and night <= XX and note not like "twilight_near_sun" '
        constraint_str = constraint_str.replace('XX','%d'%days)
    
        # leave empty if not specified
        mean_depth = {}
        min_depth_cut = {} 
        max_depth_cut = {}
       
        ##############################
        # now converts depth fluctuations to density fluctuations
        ##############################
        metric = NestedLinearMultibandModelMetric(
            density_tomography_model['year'+str(year)]['dlogN_dm5'], 
            extinction_cut=extinction_cut, n_filters=n_filters, # cuts going into ExgalM5WithCuts
            mean_depth=mean_depth, min_depth_cut=min_depth_cut, max_depth_cut=max_depth_cut,
        )
        # summary metric measures total power via angular power spectra of healpix map (thus needs nside)
        # _but_ has a bin-dependent lmax to consider same scales to consider the same scales as a fct of redshift
        summary_metrics = [
            TomographicClusteringSigma8biasMetric(
                density_tomography_model['year'+str(year)], convert_to_sigma8=convert_to_sigma8,
                power_multiplier=percentage_uncorrected, lmin=lmin),
        ]
        # then standard way of packing MetricBundles into a MetricBundleGroup
        depth_map_bundles = [maf.MetricBundle(
            metric=metric,
            slicer=slicer,
            constraint=constraint_str,
            run_name=run_name,
            summary_metrics=summary_metrics
        )]
        bd = maf.metricBundles.make_bundles_dict_from_list(depth_map_bundles)
        bgroup = maf.MetricBundleGroup(bd, opsim_fname)
        bgroup.run_all()

        # compute bias
        # should probably also return fsky
        results_sigma8_squared_bias[iy] = depth_map_bundles[0].summary_values['TomographicClusteringSigma8bias']
        all_depth_map_bundles.append(depth_map_bundles[0])

    return results_sigma8_squared_bias, all_depth_map_bundles

In [ ]:
opsim_fname = "/pscratch/sd/q/qhang/rubin_baseline_db/baseline_v3.3_10yrs.db"
run_name = splitext(basename(opsim_fname))[0]
print(opsim_fname, run_name)

percentage_uncorrected = 0.1

years = [1, 2, 4, 7, 10]

results_sigma8_squared_bias, depth_map_bundles = extract_sigma8_tomography_metric(
    opsim_fname, run_name,
    years, 
    percentage_uncorrected,
    DENSITY_TOMOGRAPHY_MODEL,
    mag_range_tolerated=1.0,
    nside = 32,
    lmin = 10, n_filters = 6, extinction_cut = 0.2,
    convert_to_sigma8=True
)


In [ ]:
results_sigma8_squared_bias

In [ ]:
for i, year in enumerate(years):
    data = depth_map_bundles[i].metric_values.data
    badval_arr = np.repeat(hp.UNSEEN, 5)
    data_slice_list = [
        badval_arr if x is None else x for x in data.tolist()
    ]
    # should be (nbins, npix)
    data_slice_arr = np.asarray(data_slice_list, dtype=float).T
    hp.mollview(data_slice_arr[0])

In [ ]:
data = data_slice_arr[0, :]
ind = np.isfinite(data)
data[~ind] = hp.UNSEEN
data = hp.remove_monopole(data, verbose=False)
data = hp.remove_dipole(data, verbose=False)
cells = hp.anafast(data)
ells = np.arange(cells.size)
lmin=10
lmax=63

polynomial_model = np.poly1d(DENSITY_TOMOGRAPHY_MODEL['year1']["poly1d_coefs_loglog"][0, :])
cells_model = np.exp(polynomial_model(np.log(ells)))

plt.plot(cells_model* (2*ells+1))
plt.plot(cells* (2*ells+1))
plt.yscale('log')

arr = cells * (2*ells+1)
np.sum(arr[lmin:lmax])

In [ ]:
stop

In [ ]:
%rm /pscratch/sd/b/bleis89/ObsStrat/code/meanz_uniformity/*.npz

In [ ]:
sim_list = [
   '/pscratch/sd/r/rhlozek/rubin_sim_data/sim_baseline/baseline_v3.3_10yrs.db',
   '/pscratch/sd/r/rhlozek/rubin_sim_data/noroll/noroll_v3.3_10yrs.db',
   '/pscratch/sd/b/beckermr/v3.4_sims_rubin/roll_uniform_mjdp0_v3.4_10yrs.db',
  # '/pscratch/sd/b/beckermr/v3.4_sims_rubin/baseline_v3.4_10yrs.db'
]
name_list = [splitext(basename(sim))[0] for sim in sim_list]

years = range(1, 11)  # [1, 2, 4, 7, 10]#
percentage_uncorrected = 0.1

results_fsky = {}
results_sigma8_squared_bias = {}
for opsim_fname, run_name in zip(sim_list, name_list):
    
    print('run_name:', run_name)
    results_sigma8_squared_bias[run_name], _ = extract_sigma8_tomography_metric(
        opsim_fname, run_name,
        years, 
        percentage_uncorrected,
        DENSITY_TOMOGRAPHY_MODEL,
        nside=64,
        lmin=10, n_filters=6, extinction_cut=0.2, mag_range_tolerated=2.0,
        convert_to_sigma8=True
    )
    
    

In [ ]:
# large mag_range_tolerated and no min depth in order to make comparison fair between strategies etc

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(7, 7), sharex=True)

colors = ['orange', 'blue', 'black', 'red']
for i, run_name in enumerate(results_sigma8_squared_bias.keys()):
    axs[0].plot(years, results_sigma8_squared_bias[run_name], label=run_name, marker='o', color=colors[i])
axs[0].legend()

results_sigma8_squared_bias.keys()
axs[1].set_xlabel('Years')
axs[0].set_ylabel('Bias in sigma8 in units of sigmas')

run_name_ = list(results_sigma8_squared_bias.keys())[0]
for i, run_name in enumerate(results_sigma8_squared_bias.keys()):
    axs[1].plot(years, np.array(years)*0, ls='--', c='orange')
    if run_name != run_name_:
        axs[1].plot(years, results_sigma8_squared_bias[run_name]-results_sigma8_squared_bias[run_name_], label=run_name, marker='o', color=colors[i])
axs[1].set_ylabel('Top panel minus '+run_name_)

In [ ]:
%rm /global/homes/b/bleis89/ObsStrat/code/meanz_uniformity/*.npz

In [ ]:
stop